# Telemetry Analysis Pipeline - MVP Testing

This notebook tests the MVP implementation of the telemetry analysis pipeline (Steps 1-5).

**Pipeline Steps:**
1. **Data Loading & Cleaning** - Load and validate telemetry data
2. **Baseline Computation** - Calculate historical percentile baselines
3. **Signal Evaluation** - Score each signal against baselines
4. **Component Aggregation** - Aggregate signals to component level
5. **Machine Aggregation** - Aggregate components to machine level
6. **Output Generation** - Write Golden layer files

**Test Data:**
- Client: `cda`
- Evaluation Week: 50
- Evaluation Year: 2025

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from datetime import datetime

# Import telemetry modules
from src.telemetry import data_loader, data_cleaner, baseline, scoring, aggregation, output_writer
from src.utils.logger import logger

print("✓ All modules imported successfully")

✓ All modules imported successfully


## Configuration

Set pipeline parameters for testing.

In [2]:
# Pipeline configuration
CLIENT = 'cda'
EVALUATION_WEEK = 50
EVALUATION_YEAR = 2025
LOOKBACK_DAYS = 7*16
BASELINE_VERSION = datetime.now().strftime('%Y%m%d')

print(f"Configuration:")
print(f"  Client: {CLIENT}")
print(f"  Evaluation: Week {EVALUATION_WEEK}, Year {EVALUATION_YEAR}")
print(f"  Baseline lookback: {LOOKBACK_DAYS} days")
print(f"  Baseline version: {BASELINE_VERSION}")

Configuration:
  Client: cda
  Evaluation: Week 50, Year 2025
  Baseline lookback: 112 days
  Baseline version: 20260225


---
## Step 1: Data Loading & Cleaning

Load the evaluation week data and apply cleaning pipeline.

In [ ]:
# Step 1A: Load evaluation week
print("=" * 60)
print("STEP 1A: Loading Evaluation Week")
print("=" * 60)

current_df = data_loader.load_evaluation_week(
    client=CLIENT,
    week=EVALUATION_WEEK,
    year=EVALUATION_YEAR
)

print(f"\n✓ Loaded {len(current_df)} rows")
print(f"  Date range: {current_df['Fecha'].min()} to {current_df['Fecha'].max()}")
print(f"  Units: {current_df['Unit'].nunique()}")
print(f"\nFirst few rows:")
current_df.head()

STEP 1A: Loading Evaluation Week
2026-02-25 15:16:50,009 - telemetry - INFO - Loading evaluation week: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\silver\cda\Telemetry_Wide_With_States\Week50Year2025.parquet


In [ ]:
# Step 1B: Get signal columns
signal_cols = data_loader.get_signal_columns(current_df)

print(f"✓ Identified {len(signal_cols)} signal columns")
print(f"\nSignal columns: {signal_cols[:10]}...")  # Show first 10

✓ Identified 18 signal columns

Signal columns: ['AirFltr', 'CnkcasePres', 'DiffLubePres', 'DiffTemp', 'EngCoolTemp', 'EngOilFltr', 'EngOilPres', 'LtExhTemp', 'LtFBrkTemp', 'LtRBrkTemp']...


In [ ]:
# Step 1C: Clean data
print("\n" + "=" * 60)
print("STEP 1C: Cleaning Data")
print("=" * 60)

current_df_clean = data_cleaner.clean_telemetry_data(current_df, signal_cols)

print(f"\n✓ Cleaning complete")
print(f"  Final rows: {len(current_df_clean)}")
print(f"  Rows removed: {len(current_df) - len(current_df_clean)}")


STEP 1C: Cleaning Data
2026-02-23 23:01:44,192 - telemetry - INFO - Starting data cleaning pipeline
2026-02-23 23:01:44,215 - telemetry - INFO - Timestamp validation: 81902 valid rows
2026-02-23 23:01:44,230 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(100.0)}
2026-02-23 23:01:44,244 - telemetry - WARNING - Flagged 26 extreme outliers as NaN
2026-02-23 23:01:44,245 - telemetry - INFO - Data cleaning complete: 81902 rows (0.0% removed)

✓ Cleaning complete
  Final rows: 81902
  Rows removed: 0


In [ ]:
# Step 1D: Load component mapping
print("\n" + "=" * 60)
print("STEP 1D: Loading Component Mapping")
print("=" * 60)

component_mapping = data_loader.load_component_mapping(CLIENT)

print(f"\n✓ Loaded mapping for {len(component_mapping)} components")
for comp_name, comp_config in list(component_mapping.items())[:5]:
    print(f"  - {comp_name}: {len(comp_config['signals'])} signals, criticality={comp_config['criticality']}")


STEP 1D: Loading Component Mapping
2026-02-23 23:01:44,253 - telemetry - INFO - Loading component mapping from c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\component_signals_mapping.json
2026-02-23 23:01:44,254 - telemetry - INFO - Loaded mapping for 4 components

✓ Loaded mapping for 4 components
  - Motor: 9 signals, criticality=High
  - Tren de fuerza: 4 signals, criticality=High
  - Frenos: 4 signals, criticality=Medium
  - Direccion: 1 signals, criticality=Medium


---
## Step 2: Baseline Computation

Load historical data and compute percentile baselines.

In [ ]:
# Step 2A: Load baseline training window
print("\n" + "=" * 60)
print("STEP 2A: Loading Baseline Training Window")
print("=" * 60)

baseline_training_df = data_loader.load_baseline_training_window(
    client=CLIENT,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    lookback_days=LOOKBACK_DAYS
)

print(f"\n✓ Loaded {len(baseline_training_df)} historical rows")
print(f"  Date range: {baseline_training_df['Fecha'].min()} to {baseline_training_df['Fecha'].max()}")
print(f"  Units: {baseline_training_df['Unit'].nunique()}")


STEP 2A: Loading Baseline Training Window
2026-02-23 23:01:44,265 - telemetry - INFO - Loading baseline data from 2025-08-25 to 2025-12-14
2026-02-23 23:01:45,714 - telemetry - INFO - Loaded 1247220 rows, 11 units for baseline training

✓ Loaded 1247220 historical rows
  Date range: 2025-08-25 00:00:00 to 2025-12-14 00:00:00
  Units: 11


In [ ]:
# Step 2B: Clean baseline data
print("\n" + "=" * 60)
print("STEP 2B: Cleaning Baseline Data")
print("=" * 60)

baseline_training_clean = data_cleaner.clean_telemetry_data(baseline_training_df, signal_cols)

print(f"\n✓ Baseline data cleaned")
print(f"  Final rows: {len(baseline_training_clean)}")


STEP 2B: Cleaning Baseline Data
2026-02-23 23:01:45,741 - telemetry - INFO - Starting data cleaning pipeline
2026-02-23 23:01:45,921 - telemetry - INFO - Timestamp validation: 1247220 valid rows
2026-02-23 23:01:46,186 - telemetry - WARNING - Removed 9597 duplicate readings
2026-02-23 23:01:46,338 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(99.15555867982415)}
2026-02-23 23:01:46,821 - telemetry - WARNING - Flagged 2825 extreme outliers as NaN
2026-02-23 23:01:46,822 - telemetry - INFO - Data cleaning complete: 1237623 rows (0.8% removed)

✓ Baseline data cleaned
  Final rows: 1237623


In [ ]:
# Step 2C: Compute baseline percentiles
print("\n" + "=" * 60)
print("STEP 2C: Computing Baseline Percentiles")
print("=" * 60)

baseline_df = baseline.compute_baseline_percentiles(
    training_df=baseline_training_clean,
    signal_cols=signal_cols,
    baseline_date=BASELINE_VERSION
)

print(f"\n✓ Computed {len(baseline_df)} baseline combinations")
print(f"\nBaseline summary:")
baseline_df.head(10)


STEP 2C: Computing Baseline Percentiles
2026-02-23 23:01:46,830 - telemetry - INFO - Computing baseline percentiles for 18 signals
2026-02-23 23:01:46,830 - telemetry - INFO -   Percentiles: [0.02, 0.05, 0.95, 0.98]
2026-02-23 23:01:46,846 - telemetry - INFO -   Training data: 1237623 rows, 11 units
2026-02-23 23:01:53,432 - telemetry - INFO - Computed 932 baseline combinations
2026-02-23 23:01:53,433 - telemetry - INFO -   Units with baselines: 11
2026-02-23 23:01:53,434 - telemetry - INFO -   Signals with baselines: 18
2026-02-23 23:01:53,436 - telemetry - INFO -   State-specific: 932, Aggregate: 0

✓ Computed 932 baseline combinations

Baseline summary:


,Unit,Signal,EstadoMaquina,P2,P5,P95,P98,sample_count,baseline_version
0,T_10,AirFltr,ND,0.375000,0.583385,4.455224,4.891893,90184,20260223
1,T_10,CnkcasePres,ND,-0.385116,-0.307461,0.253175,0.286122,114592,20260223
2,T_10,DiffLubePres,ND,24.550885,33.600170,114.447415,119.890552,82623,20260223
3,T_10,DiffTemp,ND,49.112277,62.120989,84.063095,85.500000,114843,20260223
4,T_10,EngCoolTemp,ND,63.750000,72.000000,84.875000,86.225000,114627,20260223
5,T_10,EngOilPres,ND,347.702555,363.295760,487.980833,497.752406,113812,20260223
6,T_10,LtExhTemp,ND,147.812798,166.501319,556.557985,570.602244,114859,20260223
7,T_10,LtFBrkTemp,ND,57.848124,70.583333,89.875000,92.138961,114813,20260223
8,T_10,LtRBrkTemp,ND,58.000000,70.583333,90.213992,92.602941,114819,20260223
9,T_10,RAftrclrTemp,ND,21.000000,24.958333,66.125000,69.818194,114710,20260223


In [ ]:
# Step 2D: Save baseline
print("\n" + "=" * 60)
print("STEP 2D: Saving Baseline")
print("=" * 60)

baseline_path = baseline.save_baseline(baseline_df, CLIENT)

print(f"\n✓ Baseline saved to: {baseline_path}")


STEP 2D: Saving Baseline
2026-02-23 23:01:53,467 - telemetry - INFO - Saved baseline to c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_20260223.parquet
2026-02-23 23:01:53,467 - telemetry - INFO -   Baseline version: 20260223
2026-02-23 23:01:53,467 - telemetry - INFO -   Total records: 932

✓ Baseline saved to: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_20260223.parquet


---
## Step 3: Signal Evaluation

Score each signal against baseline percentiles.

In [ ]:
# Step 3: Evaluate signals
print("\n" + "=" * 60)
print("STEP 3: Signal Evaluation")
print("=" * 60)

signal_evaluation_df = scoring.evaluate_signals(
    current_df=current_df_clean,
    baseline_df=baseline_df,
    signal_cols=signal_cols,
    component_mapping=component_mapping
)

print(f"\n✓ Evaluated {len(signal_evaluation_df)} signal-unit combinations")
print(f"\nStatus distribution:")
print(signal_evaluation_df['signal_status'].value_counts(normalize=True))

print(f"\nSample evaluations:")
signal_evaluation_df.head(10)


STEP 3: Signal Evaluation
2026-02-23 23:01:53,477 - telemetry - INFO - Starting signal evaluation
2026-02-23 23:01:53,479 - telemetry - INFO -   Units to evaluate: 11
2026-02-23 23:01:53,480 - telemetry - INFO -   Signals to evaluate: 18
2026-02-23 23:01:55,119 - telemetry - INFO - Signal evaluation complete: 184 signal-unit combinations
2026-02-23 23:01:55,121 - telemetry - INFO -   Status distribution: {'Normal': 119, 'Alerta': 40, 'Anormal': 24, 'InsufficientData': 1}

✓ Evaluated 184 signal-unit combinations

Status distribution:
signal_status
Normal              0.646739
Alerta              0.217391
Anormal             0.130435
InsufficientData    0.005435
Name: proportion, dtype: float64

Sample evaluations:


,unit_id,signal_name,component,window_score_normalized,signal_status,sample_count,anomaly_count,anomaly_percentage,max_score,p2,p5,p95,p98
0,T_10,AirFltr,Motor,0.581017,Alerta,6511,1795,27.568730,3.0,0.375000,0.583385,4.455224,4.891893
1,T_10,CnkcasePres,Motor,0.149696,Normal,7729,717,9.276750,3.0,-0.385116,-0.307461,0.253175,0.286122
2,T_10,DiffLubePres,Tren de fuerza,0.209457,Normal,5414,620,11.451792,3.0,24.550885,33.600170,114.447415,119.890552
3,T_10,DiffTemp,Tren de fuerza,0.229151,Normal,7746,963,12.432223,3.0,49.112277,62.120989,84.063095,85.500000
4,T_10,EngCoolTemp,Motor,0.202509,Normal,7733,826,10.681495,3.0,63.750000,72.000000,84.875000,86.225000
5,T_10,EngOilPres,Motor,0.164439,Normal,7705,721,9.357560,3.0,347.702555,363.295760,487.980833,497.752406
6,T_10,LtExhTemp,Motor,0.186371,Normal,7748,780,10.067114,3.0,147.812798,166.501319,556.557985,570.602244
7,T_10,LtFBrkTemp,Frenos,0.182511,Normal,7742,821,10.604495,3.0,57.848124,70.583333,89.875000,92.138961
8,T_10,LtRBrkTemp,Frenos,0.169573,Normal,7743,753,9.724913,3.0,58.000000,70.583333,90.213992,92.602941
9,T_10,RAftrclrTemp,Motor,0.138778,Normal,7739,652,8.424861,3.0,21.000000,24.958333,66.125000,69.818194


In [ ]:
# Analyze signal evaluation results
print("Signal Evaluation Analysis:")
print(f"\nTop 10 signals with highest anomaly rates:")
top_anomalies = signal_evaluation_df.nlargest(10, 'anomaly_percentage')[
    ['unit_id', 'signal_name', 'component', 'signal_status', 'anomaly_percentage', 'window_score_normalized']
]
top_anomalies

Signal Evaluation Analysis:

Top 10 signals with highest anomaly rates:


,unit_id,signal_name,component,signal_status,anomaly_percentage,window_score_normalized
137,T_18,DiffLubePres,Tren de fuerza,Anormal,69.723210,1.916654
141,T_18,EngOilPres,Motor,Anormal,59.577610,1.540688
142,T_18,LtExhTemp,Motor,Anormal,55.265500,1.486206
145,T_18,RtExhTemp,Motor,Anormal,54.420053,1.482794
34,T_12,AirFltr,Motor,Anormal,51.368180,1.411273
21,T_11,EngCoolTemp,Motor,Alerta,50.599718,0.883467
119,T_17,AirFltr,Motor,Anormal,49.616670,1.347356
131,T_17,RtLtExhTemp,Motor,Anormal,48.430214,1.212931
85,T_15,AirFltr,Motor,Anormal,47.000618,1.276850
124,T_17,EngOilPres,Motor,Anormal,45.868726,1.001081


---
## Step 4: Component Aggregation

Aggregate signal scores to component level.

In [ ]:
# Step 4: Aggregate to components
print("\n" + "=" * 60)
print("STEP 4: Component Aggregation")
print("=" * 60)

component_df = aggregation.aggregate_to_components(
    signal_evaluation_df=signal_evaluation_df,
    component_mapping=component_mapping,
    current_df=current_df_clean,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    baseline_version=BASELINE_VERSION
)

print(f"\n✓ Aggregated to {len(component_df)} component-unit combinations")
print(f"\nComponent status distribution:")
print(component_df['component_status'].value_counts())

print(f"\nSample component evaluations:")
component_df.head(10)


STEP 4: Component Aggregation
2026-02-23 23:01:55,158 - telemetry - INFO - Aggregating signals to components
2026-02-23 23:01:55,224 - telemetry - INFO - Component aggregation complete: 44 component-unit combinations
2026-02-23 23:01:55,225 - telemetry - INFO -   Status distribution: {'Normal': 32, 'Alerta': 7, 'Anormal': 5}

✓ Aggregated to 44 component-unit combinations

Component status distribution:
component_status
Normal     32
Alerta      7
Anormal     5
Name: count, dtype: int64

Sample component evaluations:


,unit_id,component,component_score,component_status,triggering_signals,signals_evaluation,signal_coverage,sample_count_avg,criticality
0,T_10,Motor,0.0375,Normal,[AirFltr],"{'AirFltr': {'status': 'Alerta', 'window_score...",0.888889,6661.125,High
1,T_10,Tren de fuerza,0.0000,Normal,[],"{'DiffLubePres': {'status': 'Normal', 'window_...",1.000000,7143.500,High
2,T_10,Frenos,0.0000,Normal,[],"{'LtFBrkTemp': {'status': 'Normal', 'window_sc...",1.000000,7739.750,Medium
3,T_10,Direccion,0.0000,Normal,[],"{'StrgOilTemp': {'status': 'Normal', 'window_s...",1.000000,7739.000,Medium
4,T_11,Motor,0.4000,Alerta,"[AirFltr, EngCoolTemp, EngOilPres, LtExhTemp, ...","{'AirFltr': {'status': 'Anormal', 'window_scor...",0.888889,8175.625,High
5,T_11,Tren de fuerza,0.0750,Normal,[DiffLubePres],"{'DiffLubePres': {'status': 'Alerta', 'window_...",1.000000,7889.750,High
6,T_11,Frenos,0.0000,Normal,[],"{'LtFBrkTemp': {'status': 'Normal', 'window_sc...",1.000000,8511.500,Medium
7,T_11,Direccion,0.0000,Normal,[],"{'StrgOilTemp': {'status': 'Normal', 'window_s...",1.000000,8511.000,Medium
8,T_12,Motor,0.5250,Anormal,"[AirFltr, CnkcasePres, EngOilPres, LtExhTemp, ...","{'AirFltr': {'status': 'Anormal', 'window_scor...",0.888889,7782.375,High
9,T_12,Tren de fuerza,0.0750,Normal,[DiffLubePres],"{'DiffLubePres': {'status': 'Alerta', 'window_...",1.000000,7514.250,High


In [ ]:
# Analyze component results
print("Component Analysis:")
print(f"\nTop 10 components with highest scores:")
top_components = component_df.nlargest(10, 'component_score')[
    ['unit_id', 'component', 'component_status', 'component_score', 'triggering_signals']
]
top_components

Component Analysis:

Top 10 components with highest scores:


,unit_id,component,component_status,component_score,triggering_signals
28,T_17,Motor,Anormal,0.6625,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
32,T_18,Motor,Anormal,0.5500,"[EngCoolTemp, EngOilFltr, EngOilPres, LtExhTem..."
16,T_14,Motor,Anormal,0.5375,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
8,T_12,Motor,Anormal,0.5250,"[AirFltr, CnkcasePres, EngOilPres, LtExhTemp, ..."
20,T_15,Motor,Anormal,0.4875,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
40,T_9,Motor,Alerta,0.4500,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
4,T_11,Motor,Alerta,0.4000,"[AirFltr, EngCoolTemp, EngOilPres, LtExhTemp, ..."
33,T_18,Tren de fuerza,Alerta,0.3250,"[DiffLubePres, TrnLubeTemp]"
34,T_18,Frenos,Alerta,0.3000,"[LtFBrkTemp, LtRBrkTemp, RtFBrkTemp, RtRBrkTemp]"
12,T_13,Motor,Alerta,0.1875,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."


---
## Step 5: Machine Aggregation

Aggregate component evaluations to machine level.

In [ ]:
unit = 'T_18'

unit_components = component_df[component_df['unit_id'] == unit]
        
# Count components by status
status_counts = unit_components['component_status'].value_counts()
components_normal = status_counts.get('Normal', 0)
components_alerta = status_counts.get('Alerta', 0)
components_anormal = status_counts.get('Anormal', 0)

print(f"Unit {unit} component status counts:"
      f"\n  Normal: {components_normal}"
      f"\n  Alerta: {components_alerta}"
      f"\n  Anormal: {components_anormal}")

Unit T_18 component status counts:
  Normal: 1
  Alerta: 2
  Anormal: 1


In [ ]:
# Step 5: Aggregate to machines
print("\n" + "=" * 60)
print("STEP 5: Machine Aggregation")
print("=" * 60)

# Get expected fleet for handling missing units
from pathlib import Path
machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
expected_units = aggregation.get_expected_fleet(
    baseline_df=baseline_df,
    previous_machine_status_path=machine_status_path
)

print(f"Expected fleet size: {len(expected_units)} units")
print(f"Units with component data: {component_df['unit_id'].nunique()} units")

machine_df = aggregation.aggregate_to_machines(
    component_df=component_df,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    baseline_version=BASELINE_VERSION,
    expected_units=expected_units,
    component_mapping=component_mapping
)

print(f"\n✓ Aggregated to {len(machine_df)} machines")
print(f"\nMachine status distribution:")
print(machine_df['overall_status'].value_counts())

print(f"\nMachine evaluation results:")
machine_df.head(11)


STEP 5: Machine Aggregation
2026-02-23 23:01:55,276 - telemetry - INFO - Aggregating components to machines
2026-02-23 23:01:55,291 - telemetry - INFO - Machine aggregation complete: 11 machines evaluated
2026-02-23 23:01:55,292 - telemetry - INFO -   Overall status distribution: {'Normal': 6, 'Alerta': 4, 'Anormal': 1}
2026-02-23 23:01:55,292 - telemetry - INFO -   Normal: 6
2026-02-23 23:01:55,293 - telemetry - INFO -   Alerta: 4
2026-02-23 23:01:55,293 - telemetry - INFO -   Anormal: 1

✓ Aggregated to 11 machines

Machine status distribution:
overall_status
Normal     6
Alerta     4
Anormal    1
Name: count, dtype: int64

Machine evaluation results:


,unit_id,overall_status,machine_score,priority_score,components_normal,components_alerta,components_anormal,components_insufficient,total_components,evaluation_week,evaluation_year,baseline_version,component_details
0,T_10,Normal,0.00,0.00,4,0,0,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Normal', 's..."
1,T_11,Normal,0.30,10.30,3,1,0,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Alerta', 's..."
2,T_12,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Anormal', '..."
3,T_13,Normal,0.30,10.30,3,1,0,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Alerta', 's..."
4,T_14,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Anormal', '..."
5,T_15,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Anormal', '..."
6,T_16,Normal,0.42,20.42,2,2,0,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Alerta', 's..."
7,T_17,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Anormal', '..."
8,T_18,Anormal,1.42,121.42,1,2,1,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Anormal', '..."
9,T_24,Normal,0.00,0.00,4,0,0,0,4,50,2025,20260223,"[{'component': 'Motor', 'status': 'Normal', 's..."


In [ ]:
# Analyze machine results
print("Machine Analysis:")
print(f"\nMachines by priority (top 10 highest priority):")
top_priority = machine_df.nlargest(10, 'priority_score')[
    ['unit_id', 'overall_status', 'machine_score', 'priority_score', 
     'components_anormal', 'components_alerta', 'components_normal']
]
top_priority

Machine Analysis:

Machines by priority (top 10 highest priority):


,unit_id,overall_status,machine_score,priority_score,components_anormal,components_alerta,components_normal
8,T_18,Anormal,1.42,121.42,1,2,1
2,T_12,Alerta,1.00,101.00,1,0,3
4,T_14,Alerta,1.00,101.00,1,0,3
5,T_15,Alerta,1.00,101.00,1,0,3
7,T_17,Alerta,1.00,101.00,1,0,3
6,T_16,Normal,0.42,20.42,0,2,2
1,T_11,Normal,0.30,10.30,0,1,3
3,T_13,Normal,0.30,10.30,0,1,3
10,T_9,Normal,0.30,10.30,0,1,3
0,T_10,Normal,0.00,0.00,0,0,4


---
## Step 6: Output Generation

Write Golden layer outputs.

In [ ]:
# Step 6: Write outputs
print("\n" + "=" * 60)
print("STEP 6: Writing Golden Layer Outputs")
print("=" * 60)

output_writer.write_golden_outputs(
    machine_df=machine_df,
    component_df=component_df,
    client=CLIENT
)

print("\n✓ Golden layer outputs written successfully")


STEP 6: Writing Golden Layer Outputs
2026-02-23 23:01:55,336 - telemetry - INFO - Writing Golden layer outputs to c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda
2026-02-23 23:01:55,345 - telemetry - INFO - Wrote machine_status: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\machine_status.parquet
2026-02-23 23:01:55,346 - telemetry - INFO -   Records: 11
2026-02-23 23:01:55,363 - telemetry - INFO - Wrote classified: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\classified.parquet
2026-02-23 23:01:55,364 - telemetry - INFO -   Records: 44
2026-02-23 23:01:55,369 - telemetry - INFO - Wrote signal_evaluations: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\signal_evaluations.parquet
2026-02-23 23:01:55,370 - telemetry - INFO -   Records: 184
2026-02-23 23:01:55,370 - telemetry - INFO - Golden layer outputs complete

✓ Golden layer outputs written successfully


In [ ]:
# Write baseline metadata
output_writer.write_baseline_metadata(
    baseline_df=baseline_df,
    client=CLIENT,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    lookback_days=LOOKBACK_DAYS
)

print("✓ Baseline metadata written")

2026-02-23 23:01:55,387 - telemetry - INFO - Wrote baseline metadata: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_metadata.json
✓ Baseline metadata written


In [ ]:
# Verify time-series behavior
print("\n" + "=" * 60)
print("TIME-SERIES VERIFICATION")
print("=" * 60)

# Load classified to check historical records
classified_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'classified.parquet'
if classified_path.exists():
    classified_check = pd.read_parquet(classified_path)
    
    print(f"\nTotal historical records in classified.parquet: {len(classified_check)}")
    
    # Check unique evaluation periods
    periods = classified_check[['evaluation_year', 'evaluation_week']].drop_duplicates().sort_values(['evaluation_year', 'evaluation_week'])
    print(f"\nEvaluation periods in history: {len(periods)}")
    print(periods.to_string(index=False))
    
    # Show sample component evolution for one unit
    if not classified_check.empty:
        sample_unit = classified_check['unit_id'].iloc[0]
        sample_component = classified_check[classified_check['unit_id'] == sample_unit]['component'].iloc[0]
        
        evolution = classified_check[
            (classified_check['unit_id'] == sample_unit) & 
            (classified_check['component'] == sample_component)
        ][['evaluation_week', 'evaluation_year', 'component_status', 'component_score']].sort_values(['evaluation_year', 'evaluation_week'])
        
        print(f"\nComponent evolution example: {sample_unit} - {sample_component}")
        print(evolution.to_string(index=False))
else:
    print("No classified.parquet file found yet")

print("\n" + "=" * 60)
print("NOTE: machine_status.parquet is overwritten (latest only)")
print("      classified.parquet appends history (time-series)")
print("=" * 60)

---
## Validation & Summary

Verify outputs and generate summary statistics.

In [ ]:
# Verify output files exist
from pathlib import Path

base_dir = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT
output_files = [
    'machine_status.parquet',
    'classified.parquet'
]

print("Output File Verification:")
for file in output_files:
    file_path = base_dir / file
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {file} ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ {file} - NOT FOUND")

Output File Verification:
  ✓ machine_status.parquet (0.01 MB)
  ✓ classified.parquet (0.03 MB)
  ✓ signal_evaluations.parquet (0.02 MB)


In [ ]:
# Generate pipeline summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 60)

print(f"\nConfiguration:")
print(f"  Client: {CLIENT}")
print(f"  Evaluation: Week {EVALUATION_WEEK}, Year {EVALUATION_YEAR}")
print(f"  Baseline: {BASELINE_VERSION} ({LOOKBACK_DAYS} days lookback)")

print(f"\nData Processing:")
print(f"  Evaluation rows: {len(current_df_clean)}")
print(f"  Baseline rows: {len(baseline_training_clean)}")
print(f"  Units evaluated: {machine_df['unit_id'].nunique()}")

print(f"\nBaseline Computation:")
print(f"  Total baselines: {len(baseline_df)}")
print(f"  State-specific: {(baseline_df['EstadoMaquina'] != 'All').sum()}")
print(f"  Aggregate: {(baseline_df['EstadoMaquina'] == 'All').sum()}")

print(f"\nSignal Evaluation:")
print(f"  Total evaluations: {len(signal_evaluation_df)}")
status_counts = signal_evaluation_df['signal_status'].value_counts()
for status, count in status_counts.items():
    pct = (count / len(signal_evaluation_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\nComponent Aggregation:")
print(f"  Total components: {len(component_df)}")
comp_status_counts = component_df['component_status'].value_counts()
for status, count in comp_status_counts.items():
    pct = (count / len(component_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\nMachine Aggregation:")
print(f"  Total machines: {len(machine_df)}")
machine_status_counts = machine_df['overall_status'].value_counts()
for status, count in machine_status_counts.items():
    pct = (count / len(machine_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\n{'=' * 60}")
print("✓ PIPELINE EXECUTION COMPLETE")
print('=' * 60)


PIPELINE EXECUTION SUMMARY

Configuration:
  Client: cda
  Evaluation: Week 50, Year 2025
  Baseline: 20260223 (112 days lookback)

Data Processing:
  Evaluation rows: 81902
  Baseline rows: 1237623
  Units evaluated: 11

Baseline Computation:
  Total baselines: 932
  State-specific: 932
  Aggregate: 0

Signal Evaluation:
  Total evaluations: 184
    Normal: 119 (64.7%)
    Alerta: 40 (21.7%)
    Anormal: 24 (13.0%)
    InsufficientData: 1 (0.5%)

Component Aggregation:
  Total components: 44
    Normal: 32 (72.7%)
    Alerta: 7 (15.9%)
    Anormal: 5 (11.4%)

Machine Aggregation:
  Total machines: 11
    Normal: 6 (54.5%)
    Alerta: 4 (36.4%)
    Anormal: 1 (9.1%)

✓ PIPELINE EXECUTION COMPLETE


---
## Next Steps

The MVP pipeline is now complete and tested. You can:

1. **Review outputs** in `data/telemetry/golden/{client}/`
2. **Load outputs** for dashboard visualization
3. **Tune thresholds** in scoring and aggregation modules
4. **Extend pipeline** with Phase 2 features (LLM, LSTM, forecasting)

For production use, create a main pipeline script that orchestrates all steps.